In [157]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path(r"/kaggle/input/competitions/name-that-llm/train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

df = pd.read_csv(data_path)
df.head()

/kaggle/input/competitions/name-that-llm/train.csv


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [158]:
import torch
torch.cuda.device_count()

2

In [159]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["model"] = le.fit_transform(df["model"])
le.transform(le.classes_)
df.head()

,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",17
1,0041ac86ee93,p073,Data science communities matter because they a...,4
2,005a19083eae,p120,"In the final twilight of time, the last embers...",6
3,009319596f10,p073,Data science communities matter because they t...,14
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",10


### DeBERTa -> Tokenizer 

In [160]:
# import os
# HF_TOKEN = os.getenv("HF_TOKEN")
# print(HF_TOKEN)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
print(HF_TOKEN)


In [161]:
from transformers import ( AutoModelForSequenceClassification, AutoTokenizer)

# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoints"
NUM_LABELS = 18
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, tokens=HF_TOKEN)

model = AutoModelForSequenceClassification.from_pretrained(
       MODEL_NAME, 
       num_labels=NUM_LABELS
    )

# model.save_pretrained(LOCAL_DIR)
# tokenizer.save_pretrained(LOCAL_DIR)
model


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifie

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [162]:
# #  switch off the the trainable parameters
# for param in model.deberta.parameters():
#     param.requires_grad = False

In [163]:
# Total and trainable parameters
total_params = model.num_parameters()
trainable_params = model.num_parameters(only_trainable=True)
print(f"Total: {total_params:,} | Trainable: {trainable_params:,}")

Total: 141,908,754 | Trainable: 141,908,754


In [164]:

# Total parameters (including frozen layers)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")


Total Parameters: 141,908,754


In [165]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

141908754

In [166]:
# Trainable parameters only (layers being updated during training)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {trainable_params:,}")

Trainable Parameters: 141,908,754


In [167]:
# # Layer wise parameter
# for param in model.parameters():
#     print(param.shape)
# for name, param in model.named_parameters():
#      print(f"Layer: {name} | Shape: {param.shape}")


In [168]:
text = [
    "The proposed method is efficient.",
    "This answer provides a detailed explanation.",
    "The model generates concise responses."
]

# 	******"padding=True" ka matlab yahan batch ke longest sequence tak padding karna hai.
#    Agar padding="max_length" use karte, toh har sequence ko specified max_length tak pad kiya jata.*****
encoded = tokenizer(
    text,
    truncation=True,
    max_length=128,
    padding=True,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

torch.Size([3, 9])
torch.Size([3, 9])


In [169]:
from torch.onnx._internal.torchscript_exporter.symbolic_opset9 import dim
import torch
outputs = model(**encoded)
logits = outputs.logits
pred = torch.argmax(logits, dim=1)
pred

prob = torch.softmax(logits, dim=1)
prob

predictions = torch.argmax(prob, dim=1)
predictions

tensor([12, 12, 12])

In [170]:
import torch
from torch.utils.data import Dataset

class LLMTextData(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
        if len(self.texts) != len(self.labels):
            raise ValueError("Texts and labels are not of equal length")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.tokenizer(
            text, 
            padding=False,
            max_length = self.max_length,
            return_tensors = "pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),     # (1, 10) -> (10,)
            "attention_mask": encoded["attention_mask"].squeeze(0),   # (1, 10) -> (10,)
            "labels": torch.tensor(label, dtype=torch.long)
        }

        return item

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df["model"])

print(train_df.shape, test_df.shape)

train_datasets = LLMTextData(
    texts=train_df["text"],
    labels=train_df["model"],
    max_length=256,
    tokenizer=tokenizer
)

test_datasets = LLMTextData(
    texts=test_df["text"],
    labels=test_df["model"],
    max_length=256,
    tokenizer=tokenizer
)

train_loader = DataLoader(
    dataset=train_datasets,
    batch_size=8,
    shuffle=True,
)

test_loader = DataLoader(
    dataset=test_datasets,
    batch_size=8,
)

print(len(train_loader), len(test_loader))

338

In [172]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

cuda


In [173]:
import torch.nn as nn
from torch.optim import AdamW

model = model.to(device)

# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)



In [174]:

from tqdm.auto import tqdm

EPOCHS = 1

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_examples = 0
    correct_predictions = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch: {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        print(logits)
        loss = criterion(logits, labels)

        # Backpropagation and weight update
        loss.backward()
        optimizer.step()

        # Metrics
        batch_size = labels.size(0)
        predictions = torch.argmax(logits, dim=1)

        correct_predictions += (predictions == labels).sum().item()
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        # Update progress bar after every batch
        progress_bar.set_postfix(
            loss=f"{total_loss / total_examples:.4f}",
            accuracy=f"{correct_predictions / total_examples:.4f}"
        )

    epoch_loss = total_loss / total_examples
    epoch_acc = correct_predictions / total_examples

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.4f}"
    )


Epoch: 1/1:   0%|          | 0/338 [00:00<?, ?it/s]

tensor([[-0.0320, -0.2061, -0.0416,  0.0100, -0.3174, -0.3345, -0.1044,  0.0023,
         -0.1230,  0.2430, -0.1203,  0.1302,  0.2546, -0.0203, -0.1847,  0.1934,
          0.1116, -0.0034],
        [ 0.1033, -0.0455,  0.0807,  0.0999, -0.5269, -0.2791, -0.0690, -0.0963,
          0.0181,  0.1790, -0.2114,  0.0542,  0.0576, -0.1289, -0.0852,  0.3000,
          0.0459,  0.0844],
        [ 0.0166, -0.1040,  0.0128, -0.1191, -0.3926, -0.3806,  0.0477, -0.0457,
          0.1227,  0.0175, -0.1334,  0.1214,  0.2196, -0.1390,  0.0125,  0.2426,
          0.1543, -0.0017],
        [ 0.0511, -0.2388,  0.2491,  0.0975, -0.4827, -0.0736,  0.2349, -0.1658,
          0.1359, -0.1119, -0.0708,  0.0354,  0.1047, -0.2448, -0.1143,  0.2771,
         -0.1239,  0.1979],
        [ 0.1416, -0.3027,  0.0098, -0.1538, -0.2401, -0.2347, -0.0267, -0.1351,
          0.0424,  0.0107, -0.1652,  0.1394,  0.0956, -0.0663,  0.0807,  0.1809,
          0.0390,  0.0946],
        [ 0.1060, -0.0039,  0.0786,  0.0664, -0.36

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, recall_score
from tqdm.auto import tqdm

y_true = []
y_pred = []
total_test_batches = len(test_loader)


model.eval()
with torch.no_grad():
    progress_bar = tqdm(test_loader, desc=f"Predicting....")
    for cnt, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        cnt_ += 1
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(pred.cpu().tolist())
        # print(f"Predicting: {cnt+1}/{total_test_batches}")


print("Accuracy:", accuracy_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred, average="macro"))
        


        


In [ ]:
# for batch in train_loader:
#     # 1. Unpack your data (depending on what your Dataset __getitem__ returns)
#     input_ids = batch['input_ids']
#     attention_mask = batch['attention_mask']
#     labels = batch['labels']
#     print(input_ids.shape, attention_mask.shape, labels.shape)
#     print("*"*50)
#     print()
#     print
#     ()

# Print the total number of batches
# total_batches = len(train_loader)
# print(f"Total number of batches: {total_batches}")


In [ ]:
# single_batch = next(iter(train_loader))

# # Check the shape of the tensor (e.g., torch.Size([batch_size, max_length]))
# print(single_batch['input_ids'].shape)

In [ ]:
# batch = next(iter(train_loader))
# print("Input IDs", batch["input_ids"].shape)
# print("Attention mask", batch["attention_mask"].shape)
# print("Labels", batch["labels"].shape)

In [ ]:
# outputs = model(
#     input_ids = batch["input_ids"],
#     attention_mask = batch["attention_mask"]
# )

# print(outputs.logits.shape)

In [ ]:
# from torch.optim import AdamW
# optimizer = AdamW(
#     params=model.parameters(),
#     lr=2e-5  #Learning rate control karta hai ki optimizer parameters ko kitne scale par update kare.
# )

In [ ]:
# from transformers import AutoTokenizer

# # Point directly to your local folder path
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # Load directly from the disk
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# text = "Hello how is the microarchitecture of this overcompressive world"
# tokens = tokenizer.tokenize(text)
# print(tokens)


In [ ]:
# print(len(tokenizer.tokenize(df.iloc[0]["text"])))

In [ ]:
# text = "Ai is a cool tech."
# encoded = tokenizer(text)

# # Tokens 
# print("Tokens:")
# print(encoded.tokens())
# # Input ids
# print("Input Ids:")
# print(encoded["input_ids"])
# # attention mask (A 1 means: "This is a real word/token. Read it and process it.")
# print("Attention mask")
# print(encoded["attention_mask"])
# # tokens from ids
# print("Tokens from Ids:")
# print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))



In [ ]:
# # 2. Complete tokenization
# encoded = tokenizer(text)

# print("\nInput IDs:")
# print(encoded["input_ids"])

# print("\nAttention Mask:")
# print(encoded["attention_mask"])

# # 3. IDs ko wapas tokens mein decode karo
# print("\nTokens from IDs:")
# print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

In [ ]:
# import torch

# logits = torch.tensor([[2.0, 1.0, 0.1]])

# # Convert logits to probabilities
# probabilities = torch.softmax(logits, dim=1)

# # Find the class with the highest probability
# predicted_class = torch.argmax(probabilities, dim=1)

# print("Logits:", logits)
# print("Probabilities:", probabilities)
# print("Predicted class:", predicted_class.item())

In [ ]:
# import torch
# print(torch.__version__)
# logits = torch.tensor([ 
#     1.2, -0.4, 2.8, 0.7,
#     0.2, 0.1, -0.8, 0.5,
#     1.0, -0.3, 0.4, 0.6,
#     0.0, 0.9, -0.2, 0.3,
#     0.8, 0.1])

# pred = torch.argmax(logits)
# print(pred.item())



In [ ]:
# logits = torch.tensor([[2, 1, 0.1]])
# prob = torch.softmax(logits, dim=-1)
# print(torch.argmax(prob))
# # print(pred)

In [ ]:
# probabilities = torch.softmax(logits, dim=-1)
# predictions = torch.argmax(logits, dim=-1)
# print(probabilities, predictions)

In [ ]:
# import torch.nn as nn
# logits = torch.tensor([[2, 1, 0.1]])
# label = torch.tensor([1])
# loss_fn = nn.CrossEntropyLoss()
# loss = loss_fn(logits, label)
# print(loss.item())

In [ ]:
# import os
# from transformers import AutoTokenizer

# MODEL_NAME = "microsoft/deberta-v3-small"
# HF_TOKEN = os.getenv("HF_TOKEN")

# # 1. Download from Hugging Face
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# # 2. Define a folder name on your disk
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # 3. Save it
# tokenizer.save_pretrained(LOCAL_DIR)
# print(f"Tokenizer successfully saved to {LOCAL_DIR}")


In [ ]:
# from transformers import AutoTokenizer

# # Point directly to your local folder path
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # Load directly from the disk
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# text = "Hello how is the microarchitecture of this overcompressive world"
# tokens = tokenizer.tokenize(text)
# print(tokens)
